<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Choose metrics

**[C05-M04-L01 · Choose metrics](https://learning.nextia-ai.com/courses/ml/m04/choose-metrics/)** · C05, Machine Learning: From Problem to Reliable Model · Module 4, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure a regression with MAE and RMSE, and a classifier with precision, recall, F1, ROC AUC and average precision, and to choose the metric that matches Larkfield's decision.

| | |
|---|---|
| **Time** | About 60 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Baselines and the improvement target, from [Define a baseline](https://learning.nextia-ai.com/courses/ml/m01/define-a-baseline/). Scores and thresholds, from [Classification intuition](https://learning.nextia-ai.com/courses/ml/m02/classification-intuition/). The final 11-feature model, from [Select useful features](https://learning.nextia-ai.com/courses/ml/m03/select-useful-features/). |
| **You do not need** | The local project. The setup below downloads the data and the course's `ticket_model.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m04/choose-metrics/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M04-L01-choose-metrics/choose-metrics-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m01/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. It is the final version from Select useful features: 11 features. Later cells import it.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Regression: MAE and RMSE on five days

A regression model predicts a number, for example the tickets of one day. The **error** of one day is the real count minus the predicted count. Two metrics summarise the errors:

- **MAE** (mean absolute error): drop the sign of each error, then take the mean. It is in tickets.
- **RMSE** (root mean squared error): square each error, take the mean, then the square root. It is in tickets too, but a large error counts much more.

Here are five days. The model predicts 30, 30, 15, 30 and 30 tickets.

> **Predict.** The last day has an error of 10 tickets, the others 2 or less. Which metric will be larger, MAE or RMSE? Then run the cell.

In [ ]:
actual = np.array([30, 32, 14, 31, 40])
predicted = np.array([30, 30, 15, 30, 30])
errors = actual - predicted
print("errors:", errors)
print("MAE:", np.abs(errors).mean())
print("RMSE:", round(float(np.sqrt((errors ** 2).mean())), 2))

> **Check.** You should see `errors: [ 0  2 -1  1 10]`, `MAE: 2.8` and `RMSE: 4.6`.

By hand: MAE = (0 + 2 + 1 + 1 + 10) ÷ 5 = 2.8. RMSE = √((0 + 4 + 1 + 1 + 100) ÷ 5) = √21.2 = 4.6. The one large error gives 100 of the 106 in the sum of squares.

> **Your turn.** Compute MAE and RMSE for the first four days only (`errors[:4]`). Put them in `mae_4` and `rmse_4`, with RMSE rounded to 2 decimals. Then run the check cell.

In [ ]:
mae_4 = float(np.abs(errors[:4]).mean())
rmse_4 = round(float(np.sqrt((errors[:4] ** 2).mean())), 2)
print(mae_4, rmse_4)

In [ ]:
expect_hash('(mae_4, rmse_4)', (mae_4, rmse_4), 'dc68fec374959b9e')

Without the large error, RMSE is close to MAE. **RMSE is much larger than MAE when a few errors are large.** Choose RMSE when one large miss is much worse than several small ones (a day with 20 more tickets than planned overloads the team). Choose MAE when every ticket of error costs about the same.

## 2. Regression: the daily tickets

The model of [Regression intuition](https://learning.nextia-ai.com/courses/ml/m02/regression-intuition/) predicts the tickets per day from `is_weekend` and the month. It learns from the days before May 2026 and is tested on May and June 2026. Run the cell.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

daily = pd.read_csv("data/daily_tickets.csv", parse_dates=["date"])
daily_train = daily[daily["date"] < "2026-05-01"]
daily_test = daily[daily["date"] >= "2026-05-01"]
columns = ["is_weekend", "month"]
x_train = pd.get_dummies(daily_train[columns].astype({"month": str}), dtype=int)
x_test = pd.get_dummies(daily_test[columns].astype({"month": str}), dtype=int).reindex(columns=x_train.columns, fill_value=0)
predicted = LinearRegression().fit(x_train, daily_train["tickets"]).predict(x_test)
day_errors = daily_test["tickets"] - predicted
print("MAE:", round(mean_absolute_error(daily_test["tickets"], predicted), 2))
print("RMSE:", round(mean_squared_error(daily_test["tickets"], predicted) ** 0.5, 2))
print("largest error:", round(day_errors.abs().max(), 1), "| days with an error above 10:", (day_errors.abs() > 10).sum(), "of", len(day_errors))

> **Check.** You should see `MAE: 6.45`, `RMSE: 8.19`, `largest error: 22.2` and `days with an error above 10: 10 of 61`.

RMSE (8.19) is about 1.3 times MAE (6.45), because 10 of the 61 days have errors above 10 tickets. For staff planning, those days matter most, so RMSE is the better summary here.

## 3. Classification: four counts on ten tickets

Now the escalation model. A **threshold** turns a score into a flag: flag the ticket when its score is at least the threshold. Compare the flags with the truth, and you get four counts, the **confusion matrix**:

- **TP** (true positive): flagged and escalated. **FP** (false positive): flagged, not escalated.
- **FN** (false negative): not flagged, but escalated. **TN** (true negative): not flagged, not escalated.

Here are ten tickets, sorted by score. Three of them escalated.

> **Predict.** At threshold 0.5, how many tickets are flagged? How many of the three escalations are caught? Then run the cell.

In [ ]:
scores_10 = np.array([0.81, 0.62, 0.55, 0.43, 0.38, 0.30, 0.22, 0.15, 0.09, 0.04])
truth_10 = np.array([1, 0, 1, 0, 0, 1, 0, 0, 0, 0])
flags_10 = (scores_10 >= 0.5).astype(int)
tp = int(((flags_10 == 1) & (truth_10 == 1)).sum())
fp = int(((flags_10 == 1) & (truth_10 == 0)).sum())
fn = int(((flags_10 == 0) & (truth_10 == 1)).sum())
tn = int(((flags_10 == 0) & (truth_10 == 0)).sum())
print("TP", tp, "FP", fp, "FN", fn, "TN", tn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(f"precision {precision:.3f}, recall {recall:.3f}, F1 {f1:.3f}, accuracy {(tp + tn) / 10:.2f}")

> **Check.** You should see `TP 2 FP 1 FN 1 TN 6`, then `precision 0.667, recall 0.667, F1 0.667, accuracy 0.80`.

- **Precision** = TP ÷ (TP + FP): of the flagged tickets, the share that escalated. Here 2 ÷ 3.
- **Recall** = TP ÷ (TP + FN): of the escalated tickets, the share that was flagged. Here 2 ÷ 3.
- **F1** = 2 × precision × recall ÷ (precision + recall): one number that is high only when both are high.
- **Accuracy** = (TP + TN) ÷ all: the share of correct decisions. Here 8 ÷ 10. A rule that never flags gets 7 ÷ 10 = 0.70 on these tickets, and it catches nothing.

## 4. Two metrics without a threshold: ROC AUC and average precision

Precision, recall and F1 need a threshold. Two metrics measure the **ranking** of the scores, with no threshold:

- **ROC AUC**: take every pair of one escalated and one other ticket. ROC AUC is the share of pairs where the escalated ticket has the higher score. Here: 3 × 7 = 21 pairs. The ticket at 0.81 is above all 7 others, the one at 0.55 above 6 of them, the one at 0.30 above 4 of them. So ROC AUC = (7 + 6 + 4) ÷ 21 = 0.810. Guessing gives 0.5; a perfect ranking gives 1.0.
- **Average precision (AP)**: walk down the sorted list, and stop at each escalated ticket. Compute the precision at each stop, then take the mean. Here the stops are at ranks 1, 3 and 6: (1 ÷ 1 + 2 ÷ 3 + 3 ÷ 6) ÷ 3 = 0.722. AP is also called the area under the precision–recall curve (PR AUC).

Run the cell to get the same numbers from scikit-learn.

In [ ]:
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)

print("ROC AUC:", round(roc_auc_score(truth_10, scores_10), 3))
print("average precision:", round(average_precision_score(truth_10, scores_10), 3))

> **Check.** You should see `ROC AUC: 0.81` and `average precision: 0.722`.

## 5. The real model on the validation set

Now the final model of [Select useful features](https://learning.nextia-ai.com/courses/ml/m03/select-useful-features/): `ticket_model.py` with 11 features, fitted on train only. It scores the 1,179 May tickets.

> **Predict.** A constant score has ROC AUC 0.5. What average precision does a constant score get? Hint: every ticket ties, so the precision at every stop is the escalation rate.

In [ ]:
from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
print("ROC AUC:", round(roc_auc_score(valid[TARGET], scores), 3))
print("average precision:", round(average_precision_score(valid[TARGET], scores), 3))
print("escalation rate (the AP of a constant score):", round(valid[TARGET].mean(), 3))

> **Check.** You should see `ROC AUC: 0.752`, `average precision: 0.391` and `escalation rate (the AP of a constant score): 0.155`.

ROC AUC 0.752 sounds high, but its floor is 0.5. AP 0.391 must be compared with its own floor, 0.155: the model is about 2.5 times better than a constant score at putting escalations at the top. With only 15.5% escalated, most pairs in ROC AUC are easy (a low-risk ticket below an escalation). AP looks only at the top of the list, where the senior team works. So AP is the main ranking metric of this course.

The next cell defines a small helper, `report()`, and compares four ways to flag the May tickets. Run it.

In [ ]:
def report(name, flags):
    """Flag count, precision, recall, F1 and accuracy of 0/1 flags on valid."""
    truth = valid[TARGET]
    print(f"{name:24s} flagged {flags.sum():4d} ({flags.mean():5.1%})  precision {precision_score(truth, flags, zero_division=0):.3f}"
          f"  recall {recall_score(truth, flags):.3f}  F1 {f1_score(truth, flags, zero_division=0):.3f}  accuracy {accuracy_score(truth, flags):.3f}")

report("never escalate", np.zeros(len(valid), dtype=int))
report("rule: priority 1", (valid["priority"] == 1).astype(int))
report("model, threshold 0.5", (scores >= 0.5).astype(int))
report("model, threshold 0.19", (scores >= 0.19).astype(int))

> **Check.** You should see four lines:
>
> - never escalate: flagged 0, recall 0.000, accuracy 0.845;
> - the rule: flagged 163 (13.8%), precision 0.374, recall 0.333, F1 0.353, accuracy 0.810;
> - the model at 0.5: flagged 25 (2.1%), precision 0.680, recall 0.093, accuracy 0.852;
> - the model at 0.19: flagged 227 (19.3%), precision 0.379, recall 0.470, F1 0.420, accuracy 0.798.

Read the accuracy column. "Never escalate" (0.845) beats the model at 0.19 (0.798), and catches nothing. The model at 0.5 has the best accuracy, and catches 17 of 183 escalations. **Accuracy rewards the "not escalated" answer, because 84.5% of tickets are not escalated.** It does not measure Larkfield's decision.

## 6. Compare at the same workload

Grace's decision has a limit: the senior team can review at most 20% of new tickets. So compare models **at the same number of flags**. The rule flags 163 tickets. What does the model do with the same 163 reviews?

> **Your turn.** Flag the 163 tickets with the highest scores. Hint: `np.sort(scores)[::-1][162]` is the 163rd highest score; flag every score at or above it. Put the flags (0 or 1) in `flags_163`, then compute `precision_163` and `recall_163`, rounded to 3 decimals. Run the check cell.

In [ ]:
flags_163 = (scores >= np.sort(scores)[::-1][162]).astype(int)
precision_163 = round(precision_score(valid[TARGET], flags_163), 3)
recall_163 = round(recall_score(valid[TARGET], flags_163), 3)
print(flags_163.sum(), precision_163, recall_163)

In [ ]:
expect_hash('(precision_163, recall_163)', (precision_163, recall_163), '7c36c461311d4ae9')

With the same 163 reviews, the model is better than the rule on both precision and recall. That is a fair comparison: same workload, same tickets.

## 7. The trap: choose the threshold with the best accuracy

Tomás wants "the most accurate model". He tries every threshold from 0.01 to 0.99 and keeps the one with the best accuracy.

> **Predict.** Will the best-accuracy threshold be high (few flags) or low (many flags)? Then run the cell.

In [ ]:
rows = []
for threshold in np.arange(1, 100) / 100:
    flags = (scores >= threshold).astype(int)
    rows.append({"threshold": threshold, "accuracy": accuracy_score(valid[TARGET], flags),
                 "recall": recall_score(valid[TARGET], flags), "f1": f1_score(valid[TARGET], flags, zero_division=0),
                 "flag_rate": flags.mean()})
by_threshold = pd.DataFrame(rows)
print(by_threshold.loc[by_threshold["accuracy"].idxmax()].round(3).to_dict())
print(by_threshold.loc[by_threshold["f1"].idxmax()].round(3).to_dict())

> **Check.** The first line is the best accuracy: threshold 0.4, accuracy 0.854, recall 0.175, flag rate 0.045. The second line is the best F1: threshold 0.14, F1 0.439, flag rate 0.293.

The most accurate threshold catches 17.5% of escalations, less than the priority-1 rule (33.3%). The best-F1 threshold flags 29.3% of tickets, more than the senior team can review. **Neither metric knows about the 20% capacity.** The decision metric is: recall (and precision) at a flag rate of at most 20%. Lesson 3 chooses that threshold.

The chart shows the three curves. Run the cell.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for column, label in [("accuracy", "accuracy"), ("recall", "recall"), ("flag_rate", "share of tickets flagged")]:
    ax.plot(by_threshold["threshold"], by_threshold[column], label=label)
ax.axhline(0.2, color="grey", linestyle=":", label="capacity: 20% flagged")
ax.set_xlabel("Threshold")
ax.set_ylabel("Value on the validation set")
ax.legend()
plt.show()

> **Check.** You should see the accuracy curve rise from about 0.16 at threshold 0.01, then stay flat near 0.85 from about 0.3 on. Recall and the flag rate fall from 1.0 toward 0. The flag-rate curve crosses the capacity line near threshold 0.19.

## 8. Change one thing: rarer escalations

AP depends on how common escalations are; ROC AUC does not. Test it: keep all 996 other tickets, but only 61 of the 183 escalations (picked at random). The model and its scores do not change.

> **Predict.** Will ROC AUC go up, down or stay about the same? And AP? Then run the cell.

In [ ]:
rng = np.random.default_rng(0)
escalated = np.where(valid[TARGET] == 1)[0]
others = np.where(valid[TARGET] == 0)[0]
keep = np.concatenate([rng.choice(escalated, 61, replace=False), others])
truth_rare = valid[TARGET].to_numpy()[keep]
print(f"escalated: {truth_rare.mean():.1%}")
print("ROC AUC:", round(roc_auc_score(truth_rare, scores[keep]), 3))
print("average precision:", round(average_precision_score(truth_rare, scores[keep]), 3))

> **Check.** You should see `escalated: 5.8%`, `ROC AUC: 0.761` and `average precision: 0.163`.

ROC AUC stays about the same (0.761 instead of 0.752). AP falls from 0.391 to 0.163, because each flag now finds fewer escalations. Both are correct. So compare AP only between models on the **same** tickets, and always next to its floor, the escalation rate (here 0.058).

## Next

You can now measure a regression and a classifier, and choose the metric from the decision: recall and precision at the capacity, with AP for the ranking. The [lesson page](https://learning.nextia-ai.com/courses/ml/m04/choose-metrics/) has the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Validation strategy](https://learning.nextia-ai.com/courses/ml/m04/validation-strategy/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m04/choose-metrics/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.